# SCF Phase 2 shard 23

Sweeps: estimation. Jobs: 64. Projected: 5.0 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a599df9bde9f\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/a599df9bde9f.parquet\", \"means_path\": \"data/sim/estimation/means/a599df9bde9f.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 1600, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8fe0a3e6188f\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/8fe0a3e6188f.parquet\", \"means_path\": \"data/sim/estimation/means/8fe0a3e6188f.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 8000, \"p\": 1600, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6c306ab4e131\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 80, \"raw_path\": \"data/sim/estimation/raw/6c306ab4e131.parquet\", \"means_path\": \"data/sim/estimation/means/6c306ab4e131.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"b0fc97bfbd47\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/b0fc97bfbd47.parquet\", \"means_path\": \"data/sim/estimation/means/b0fc97bfbd47.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [1.118033988749895], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9701b0e3d3cf\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/9701b0e3d3cf.parquet\", \"means_path\": \"data/sim/estimation/means/9701b0e3d3cf.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [6.708203932499369], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9a018916c2fb\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/9a018916c2fb.parquet\", \"means_path\": \"data/sim/estimation/means/9a018916c2fb.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 1, \"l\": [6.708203932499369], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"71b23f7c8f7d\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/71b23f7c8f7d.parquet\", \"means_path\": \"data/sim/estimation/means/71b23f7c8f7d.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"662803b9f9a1\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/662803b9f9a1.parquet\", \"means_path\": \"data/sim/estimation/means/662803b9f9a1.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"4811cfe1b652\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/4811cfe1b652.parquet\", \"means_path\": \"data/sim/estimation/means/4811cfe1b652.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"b2bb511dea4a\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/b2bb511dea4a.parquet\", \"means_path\": \"data/sim/estimation/means/b2bb511dea4a.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 5, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"f01a43230368\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/f01a43230368.parquet\", \"means_path\": \"data/sim/estimation/means/f01a43230368.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 25, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"2234890e1fa2\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/2234890e1fa2.parquet\", \"means_path\": \"data/sim/estimation/means/2234890e1fa2.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 25, \"l\": [1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"48429ed2e87e\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/48429ed2e87e.parquet\", \"means_path\": \"data/sim/estimation/means/48429ed2e87e.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 25, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"3919b2a463c8\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/3919b2a463c8.parquet\", \"means_path\": \"data/sim/estimation/means/3919b2a463c8.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 2500, \"r\": 25, \"l\": [6.708203932499369, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895, 1.118033988749895], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"074c37374435\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/074c37374435.parquet\", \"means_path\": \"data/sim/estimation/means/074c37374435.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"5b89dceca0cb\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/5b89dceca0cb.parquet\", \"means_path\": \"data/sim/estimation/means/5b89dceca0cb.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"5e66c5030c2c\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/5e66c5030c2c.parquet\", \"means_path\": \"data/sim/estimation/means/5e66c5030c2c.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [4.242640687119286], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"76c805b0083d\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/76c805b0083d.parquet\", \"means_path\": \"data/sim/estimation/means/76c805b0083d.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 1, \"l\": [4.242640687119286], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"451651794b1c\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/451651794b1c.parquet\", \"means_path\": \"data/sim/estimation/means/451651794b1c.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"c29a39d94d81\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/c29a39d94d81.parquet\", \"means_path\": \"data/sim/estimation/means/c29a39d94d81.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"6b6214a1a1a6\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/6b6214a1a1a6.parquet\", \"means_path\": \"data/sim/estimation/means/6b6214a1a1a6.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9f8e1a122970\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/9f8e1a122970.parquet\", \"means_path\": \"data/sim/estimation/means/9f8e1a122970.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 5, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"04b6f33a9bdf\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/04b6f33a9bdf.parquet\", \"means_path\": \"data/sim/estimation/means/04b6f33a9bdf.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 25, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"530a8fa8154f\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/530a8fa8154f.parquet\", \"means_path\": \"data/sim/estimation/means/530a8fa8154f.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 25, \"l\": [0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"a40d27ec1470\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/a40d27ec1470.parquet\", \"means_path\": \"data/sim/estimation/means/a40d27ec1470.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 25, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"9da42d209cad\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/9da42d209cad.parquet\", \"means_path\": \"data/sim/estimation/means/9da42d209cad.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 1000, \"r\": 25, \"l\": [4.242640687119286, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476, 0.7071067811865476], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": true}, \"config_id\": \"0127e799a7b2\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/0127e799a7b2.parquet\", \"means_path\": \"data/sim/estimation/means/0127e799a7b2.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 3, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"conf_kind\": \"sparse\", \"beta_kind\": \"aligned\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"c07e634ddc0b\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 300, \"raw_path\": \"data/sim/estimation/raw/c07e634ddc0b.parquet\", \"means_path\": \"data/sim/estimation/means/c07e634ddc0b.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"62cddeed3492\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/62cddeed3492.parquet\", \"means_path\": \"data/sim/estimation/means/62cddeed3492.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b3baa3ee5fa3\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/b3baa3ee5fa3.parquet\", \"means_path\": \"data/sim/estimation/means/b3baa3ee5fa3.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [2.6832815729997477], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"db1e309cc881\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/db1e309cc881.parquet\", \"means_path\": \"data/sim/estimation/means/db1e309cc881.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [2.6832815729997477], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8ca8e45df894\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/8ca8e45df894.parquet\", \"means_path\": \"data/sim/estimation/means/8ca8e45df894.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b862e64374ff\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/b862e64374ff.parquet\", \"means_path\": \"data/sim/estimation/means/b862e64374ff.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"7697b0bfc460\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/7697b0bfc460.parquet\", \"means_path\": \"data/sim/estimation/means/7697b0bfc460.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6a1f0821f0ba\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/6a1f0821f0ba.parquet\", \"means_path\": \"data/sim/estimation/means/6a1f0821f0ba.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b5cf0f45c812\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/b5cf0f45c812.parquet\", \"means_path\": \"data/sim/estimation/means/b5cf0f45c812.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 25, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"528e284736f5\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/528e284736f5.parquet\", \"means_path\": \"data/sim/estimation/means/528e284736f5.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 25, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"08f648768f58\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/08f648768f58.parquet\", \"means_path\": \"data/sim/estimation/means/08f648768f58.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2cda696a519c\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/2cda696a519c.parquet\", \"means_path\": \"data/sim/estimation/means/2cda696a519c.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5b5d53f5fcc4\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/5b5d53f5fcc4.parquet\", \"means_path\": \"data/sim/estimation/means/5b5d53f5fcc4.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6ebf3dff64e0\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/6ebf3dff64e0.parquet\", \"means_path\": \"data/sim/estimation/means/6ebf3dff64e0.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"fe4a5fd53f23\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/fe4a5fd53f23.parquet\", \"means_path\": \"data/sim/estimation/means/fe4a5fd53f23.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"fbda8a9770d0\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/fbda8a9770d0.parquet\", \"means_path\": \"data/sim/estimation/means/fbda8a9770d0.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5d3742033900\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/5d3742033900.parquet\", \"means_path\": \"data/sim/estimation/means/5d3742033900.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8498b7e148cf\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/8498b7e148cf.parquet\", \"means_path\": \"data/sim/estimation/means/8498b7e148cf.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"0ae859e1e16e\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/0ae859e1e16e.parquet\", \"means_path\": \"data/sim/estimation/means/0ae859e1e16e.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"aa5f5b8c7087\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/aa5f5b8c7087.parquet\", \"means_path\": \"data/sim/estimation/means/aa5f5b8c7087.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6b4455fa7c3c\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/6b4455fa7c3c.parquet\", \"means_path\": \"data/sim/estimation/means/6b4455fa7c3c.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 25, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5332c3907759\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/5332c3907759.parquet\", \"means_path\": \"data/sim/estimation/means/5332c3907759.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 25, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"238c216d9edb\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/238c216d9edb.parquet\", \"means_path\": \"data/sim/estimation/means/238c216d9edb.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f183955c8352\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/f183955c8352.parquet\", \"means_path\": \"data/sim/estimation/means/f183955c8352.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4d6145f616e8\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 200, \"raw_path\": \"data/sim/estimation/raw/4d6145f616e8.parquet\", \"means_path\": \"data/sim/estimation/means/4d6145f616e8.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"04fc097446d7\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/04fc097446d7.parquet\", \"means_path\": \"data/sim/estimation/means/04fc097446d7.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2590da88f733\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/2590da88f733.parquet\", \"means_path\": \"data/sim/estimation/means/2590da88f733.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e39c143e8462\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/e39c143e8462.parquet\", \"means_path\": \"data/sim/estimation/means/e39c143e8462.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"99da69f3f06e\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/99da69f3f06e.parquet\", \"means_path\": \"data/sim/estimation/means/99da69f3f06e.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d7579875aa48\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/d7579875aa48.parquet\", \"means_path\": \"data/sim/estimation/means/d7579875aa48.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"3bc5eedd9102\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/3bc5eedd9102.parquet\", \"means_path\": \"data/sim/estimation/means/3bc5eedd9102.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"72f1076da848\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/72f1076da848.parquet\", \"means_path\": \"data/sim/estimation/means/72f1076da848.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f1c6ffdb910d\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/f1c6ffdb910d.parquet\", \"means_path\": \"data/sim/estimation/means/f1c6ffdb910d.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 25, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"eeefce43f701\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/eeefce43f701.parquet\", \"means_path\": \"data/sim/estimation/means/eeefce43f701.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 25, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"46b699cde0b9\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/46b699cde0b9.parquet\", \"means_path\": \"data/sim/estimation/means/46b699cde0b9.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8ed9457cbda3\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/8ed9457cbda3.parquet\", \"means_path\": \"data/sim/estimation/means/8ed9457cbda3.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"eed13c25981d\", \"mode\": \"estimation\", \"sweep\": \"estimation\", \"reps\": 140, \"raw_path\": \"data/sim/estimation/raw/eed13c25981d.parquet\", \"means_path\": \"data/sim/estimation/means/eed13c25981d.npz\", \"_rank\": 2, \"_sweep\": \"estimation\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 23, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(23), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)